In [17]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()


In [24]:
import requests
import pandas as pd
import certifi
import yfinance as yf
import math
import ta


url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; ResearchNotebook/1.0)"
}

response = requests.get(
    url,
    headers=headers,
    verify=certifi.where(),
    timeout=10
)
response.raise_for_status()

tables = pd.read_html(response.text)
sp500 = tables[0]

TICKERS = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()

len(TICKERS), TICKERS[:10]
START_DATE = "2018-01-01"

/var/folders/wz/vg2971s50r937v89lnqmjxdh0000gn/T/ipykernel_30295/1443508198.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [25]:
def load_weekly_data(ticker):
    df = yf.Ticker(ticker).history(
        start=START_DATE,
        interval="1d",
        auto_adjust=True
    )

    # Keep only Close and force Series
    close = df["Close"].astype(float)

    # Weekly resample
    close = close.resample("W-FRI").last().dropna()

    return pd.DataFrame({"Close": close})


In [26]:
def compute_score(df):
    close = df["Close"].squeeze()  # ensures 1D Series
    ma40 = close.rolling(40).mean()
    rsi = ta.momentum.RSIIndicator(close, window=14).rsi()
    
    score = (
        (close > ma40).astype(int) * 2 +
        (rsi < 70).astype(int) * 1 +
        ((close / close.shift(12)) - 1) * 10
    )
    
    return score.iloc[-1]


In [27]:
def batch_download(tickers, start_date=START_DATE):
    batch_size = 50
    all_data = {}
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        print(f"Downloading batch {i//batch_size+1}/{math.ceil(len(tickers)/batch_size)}...")
        data = yf.download(batch, start=start_date, interval="1d", auto_adjust=True, progress=False)
        
        # If only one ticker, data["Close"] is Series, wrap in DataFrame
        if len(batch) == 1:
            df_close = data["Close"].to_frame(name="Close")
            all_data[batch[0]] = df_close
        else:
            # Multiple tickers: MultiIndex, extract each ticker
            for t in batch:
                df_close = data["Close"][t].dropna().to_frame(name="Close")
                all_data[t] = df_close
    
    return all_data


In [29]:
all_data = batch_download(TICKERS)

candidates = []

for ticker, df in all_data.items():
    if len(df) < 52:  # at least 1 year of weekly data
        continue
    try:
        score = compute_score(df)
        candidates.append({
            "Ticker": ticker,
            "Score": score,
            "Price": df["Close"].iloc[-1]
        })
    except Exception as e:
        print(f"Skipping {ticker}: {e}")
        continue

# Rank top 10
top10 = pd.DataFrame(candidates).sort_values("Score", ascending=False).head(10)
top10


,Ticker,Score,Price
405,SNDK,10.802552,446.334991
316,MRNA,6.350627,42.320000
489,WDC,6.145829,226.463196
277,KLAC,5.514650,1520.630005
251,INTC,5.466775,49.692402
214,GNRC,5.205031,166.440002
118,FIX,5.111884,1130.390015
77,BLDR,5.065312,124.139999
282,LRCX,5.015540,222.800003
13,ALB,4.994132,169.645004


In [30]:
top10 = (
    pd.DataFrame(candidates)
    .sort_values("Score", ascending=False)
    .head(10)
)

top10


,Ticker,Score,Price
405,SNDK,10.802552,446.334991
316,MRNA,6.350627,42.320000
489,WDC,6.145829,226.463196
277,KLAC,5.514650,1520.630005
251,INTC,5.466775,49.692402
214,GNRC,5.205031,166.440002
118,FIX,5.111884,1130.390015
77,BLDR,5.065312,124.139999
282,LRCX,5.015540,222.800003
13,ALB,4.994132,169.645004
